# Experimento 004 - Validacao da Estrutura do Dataset

Referencias externas: [TAO Image Classification PyT](https://docs.nvidia.com/tao/tao-toolkit/latest/text/cv_finetuning/pytorch/image_classification_pyt.html), [Formato Image Classification PyTorch](https://docs.nvidia.com/tao/tao-toolkit/latest/text/data_annotation_format.html#image-classification-format-pytorch), [TAO Deploy Classification PyT](https://docs.nvidia.com/tao/tao-toolkit/latest/text/tao_deploy/classification_pyt.html) e [documentacao geral do TAO](https://docs.nvidia.com/tao/tao-toolkit/latest/).

Este experimento repete o treinamento real `classification_pyt` com foco no problema historico de estrutura do dataset: o arquivo tar precisava conter as pastas raiz `images_train`, `images_val`/`images_test` e `classes.txt`.

Escopo: validacao de contexto, preflight do dataset, treino real, polling, logs, metricas e artefatos. Avaliacao, ONNX, TensorRT e inferencia ficam fora deste experimento.

## 1. Configuracao e credenciais

A chave e carregada automaticamente do `secrets.json` local. Nenhuma credencial e gravada no notebook.

In [11]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path
import requests

TAO_BASE_URL = os.getenv("TAO_BASE_URL", "http://100.107.81.126:8090").rstrip("/")
TAO_ORG = os.getenv("TAO_ORG", "getter")
WORKSPACE_ID = os.getenv("TAO_TEST_WORKSPACE_ID", "98c4bec1-88df-423e-8a15-af78881e9742")
DATASET_URI = os.getenv("TAO_TRAIN_DATASET_URI", "seaweedfs://tao-storage/data/tao-test-classification-v1")
BASE_EXPERIMENT_ID = os.getenv("TAO_BASE_EXPERIMENT_ID", "469f42fe-37dc-55ff-b89b-5096477f9a87")
EVIDENCE_DIR = Path(os.getenv("TAO_EXPERIMENT_004_EVIDENCE", "experiment-004-evidence"))
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
configured = os.getenv("TAO_SECRETS_FILE", "")
candidates = [Path(configured)] if configured else []
candidates += [Path("secrets.json"), Path.cwd() / "secrets.json", Path.cwd().parent / "secrets.json", Path("/home/getter/tao-ftms/secrets.json")]
secrets_file = next((item for item in candidates if item.exists()), None)
NGC_KEY = json.loads(secrets_file.read_text())["ngc_api_key"] if secrets_file else os.getenv("NGC_KEY", "")
if not NGC_KEY: raise RuntimeError("NGC key not found")
http = requests.Session()
print({"base_url": TAO_BASE_URL, "workspace": WORKSPACE_ID, "dataset": DATASET_URI, "evidence": str(EVIDENCE_DIR)})

{'base_url': 'http://100.107.81.126:8090', 'workspace': '98c4bec1-88df-423e-8a15-af78881e9742', 'dataset': 'seaweedfs://tao-storage/data/tao-test-classification-v1', 'evidence': 'experiment-004-evidence'}


## 2. Login e health

O FTMS precisa estar pronto antes de consultar o dataset ou criar o job.

In [12]:
def api(method, path, payload=None, headers=None):
    merged = {"Content-Type": "application/json"}
    if headers: merged.update(headers)
    response = http.request(method, TAO_BASE_URL + path, headers=merged, json=payload, timeout=(10, 120))
    response.raise_for_status()
    return response.json()

login = api("POST", "/api/v2/login", {"ngc_key": NGC_KEY, "ngc_org_name": TAO_ORG, "enable_telemetry": False}, {"ngc_key": NGC_KEY})
TOKEN = login["token"]
AUTH = {"Authorization": f"Bearer {TOKEN}"}
health = api("GET", "/api/v2/health")
assert "readiness" in health
print("login=OK health=OK")

login=OK health=OK


## 3. Preflight do dataset

Dataset classification_pyt usado nesta rodada. Documentacao completa: `docs/datasets/classification-pyt-test-v1.md`

Referencias externas: [Image Classification PyT](https://docs.nvidia.com/tao/tao-toolkit/latest/text/cv_finetuning/pytorch/image_classification_pyt.html) e [Formato Image Classification PyTorch](https://docs.nvidia.com/tao/tao-toolkit/latest/text/data_annotation_format.html#image-classification-format-pytorch).

- URI interno: `seaweedfs://tao-storage/data/tao-test-classification-v1`
- Consulta via API: `http://100.107.81.126:8090/api/v2/orgs/getter/datasets`

O URI `seaweedfs://` e interno ao FTMS/SeaweedFS; o link HTTP acima permite consultar os metadados do dataset com autenticacao. O erro historico do Experimento 004 foi um tar sem a pasta raiz `images_train`. A API retorna metadados e URI; a estrutura interna do objeto e confirmada nos logs do job, por isso registramos ambos.

In [13]:
workspace = api("GET", f"/api/v2/orgs/{TAO_ORG}/workspaces/{WORKSPACE_ID}", headers=AUTH)
datasets = api("GET", f"/api/v2/orgs/{TAO_ORG}/datasets", headers=AUTH)
schema = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:schema?network_arch=classification_pyt&action=train", headers=AUTH)
base = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs:list_base_experiments?network_arch=classification_pyt", headers=AUTH)
matches = [item for item in datasets.get("datasets", []) if item.get("cloud_file_path") == "data/tao-test-classification-v1"]
preflight = {"workspace": workspace, "dataset_matches": matches, "expected_root": DATASET_URI, "schema_properties": list(schema.get("properties", {})), "base_experiment_found": any(item.get("id") == BASE_EXPERIMENT_ID for item in base.get("experiments", [])), "expected_layout": ["images_train/", "images_val/ or images_test/", "classes.txt"]}
(EVIDENCE_DIR / "dataset-preflight.json").write_text(json.dumps(preflight, indent=2), encoding="utf-8")
print(json.dumps({"workspace_ok": workspace.get("id") == WORKSPACE_ID, "dataset_matches": len(matches), "base_experiment_found": preflight["base_experiment_found"], "expected_layout": preflight["expected_layout"]}, indent=2))
assert workspace.get("id") == WORKSPACE_ID
assert matches
assert preflight["base_experiment_found"]

{
  "workspace_ok": true,
  "dataset_matches": 1,
  "base_experiment_found": true,
  "expected_layout": [
    "images_train/",
    "images_val/ or images_test/",
    "classes.txt"
  ]
}


## 4. Payload de treino corrigido

O payload usa uma GPU, uma epoca, duas classes e o dataset validado. O `classes_file` e mantido no URI SeaweedFS para evitar o problema de metadados ausentes.

In [14]:
spec_candidates = [Path(os.getenv("TAO_TRAIN_SPECS_JSON_FILE", ""))] if os.getenv("TAO_TRAIN_SPECS_JSON_FILE") else []
spec_candidates += [Path("train_specs.json"), Path("/home/getter/tao-real-tests-20261001/train_specs.json")]
spec_file = next((item for item in spec_candidates if item.exists()), None)
specs = json.loads(spec_file.read_text()) if spec_file else schema.get("default", {})
specs.setdefault("train", {}).update({"num_epochs": 1, "num_gpus": 1})
specs.setdefault("dataset", {}).update({"num_classes": 2, "root_dir": DATASET_URI})
payload = {"kind": "experiment", "name": f"experiment-004-dataset-layout-{int(time.time())}", "network_arch": "classification_pyt", "encryption_key": "unittest-only", "workspace": WORKSPACE_ID, "action": "train", "specs": specs, "train_dataset_uris": [DATASET_URI], "eval_dataset_uri": DATASET_URI, "inference_dataset_uri": DATASET_URI, "base_experiment_ids": [BASE_EXPERIMENT_ID]}
(EVIDENCE_DIR / "train-payload.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(json.dumps({"name": payload["name"], "network_arch": payload["network_arch"], "train": specs.get("train"), "dataset": specs.get("dataset")}, indent=2))

{
  "name": "experiment-004-dataset-layout-1792439076",
  "network_arch": "classification_pyt",
  "train": {
    "num_gpus": 1,
    "gpu_ids": [
      0
    ],
    "num_nodes": 1,
    "seed": 1234,
    "cudnn": {
      "benchmark": false,
      "deterministic": true
    },
    "num_epochs": 1,
    "checkpoint_interval": 1,
    "checkpoint_interval_unit": "epoch",
    "validation_interval": 1,
    "resume_training_checkpoint_path": "",
    "results_dir": "",
    "optim": {
      "monitor_name": "val_loss",
      "optim": "adamw",
      "lr": 6e-05,
      "policy": "linear",
      "policy_params": {
        "step_size": 30,
        "gamma": 0.1
      },
      "momentum": 0.9,
      "weight_decay": 0.01,
      "betas": [
        0.9,
        0.999
      ],
      "skip_names": [],
      "warmup_epochs": 0
    },
    "pretrained_model_path": "",
    "tensorboard": {
      "enabled": false,
      "infrequent_logging_frequency": 2
    },
    "enable_ema": false,
    "ema_decay": 0.998,
    "c

## 5. Submissao e polling

Esta e a unica celula que cria o job GPU. Ela aguarda ate `Done`/`Completed`, salva status e logs e falha explicitamente em `Error`, `Failed` ou timeout.

In [15]:
submitted = api("POST", f"/api/v2/orgs/{TAO_ORG}/jobs", payload, AUTH)
JOB_ID = submitted["id"]
(EVIDENCE_DIR / "job-id.txt").write_text(JOB_ID + "\n")
deadline = time.monotonic() + int(os.getenv("TAO_JOB_TIMEOUT_SECONDS", "3600"))
while time.monotonic() < deadline:
    status = api("GET", f"/api/v2/orgs/{TAO_ORG}/jobs/{JOB_ID}", headers=AUTH)
    (EVIDENCE_DIR / "train-status.json").write_text(json.dumps(status, indent=2))
    state = status.get("status")
    details = next(iter(status.get("job_details", {}).values()), {})
    print(datetime.now(timezone.utc).isoformat(), state, details.get("detailed_status", {}).get("message"))
    if state in {"Done", "Completed"}: break
    if state in {"Error", "Failed", "Failure", "Canceled", "Cancelled"}: raise RuntimeError(state)
    time.sleep(10)
else: raise TimeoutError(JOB_ID)
logs = http.get(f"{TAO_BASE_URL}/api/v2/orgs/{TAO_ORG}/jobs/{JOB_ID}:logs", headers=AUTH, timeout=(10, 120))
if logs.ok: (EVIDENCE_DIR / "train-logs.txt").write_text(logs.text)
print("terminal_status=", status.get("status"))

2026-10-19T19:44:38.794330+00:00 Pending None
2026-10-19T19:44:48.933939+00:00 Started Docker image 'nvidia/tao/tao-toolkit:6.26.3-pyt' found locally. Starting container...
2026-10-19T19:44:59.069995+00:00 Started Docker image 'nvidia/tao/tao-toolkit:6.26.3-pyt' found locally. Starting container...
2026-10-19T19:45:09.175933+00:00 Started Download completed: 55 files, 100.1 MB in 0:00:08
2026-10-19T19:45:19.272796+00:00 Started Training loop complete.
2026-10-19T19:45:29.391059+00:00 Started train action completed successfully for classification_pyt
2026-10-19T19:45:39.529348+00:00 Started train action completed successfully for classification_pyt
2026-10-19T19:45:49.666595+00:00 Started train action completed successfully for classification_pyt
2026-10-19T19:45:59.805231+00:00 Done train action completed successfully for classification_pyt
terminal_status= Done


## 6. Metricas, layout observado e Definition of Done

O sucesso exige status terminal, status detalhado `SUCCESS`, metricas e evidencia de que o job conseguiu localizar a estrutura real do dataset.

In [16]:
details = next(iter(status.get("job_details", {}).values()), {})
metrics = details.get("kpi", [])
result = {"experiment": "004-training-real", "status": status.get("status"), "job_id": JOB_ID, "metrics": metrics, "detailed_status": details.get("detailed_status"), "evidence_dir": str(EVIDENCE_DIR)}
(EVIDENCE_DIR / "summary.json").write_text(json.dumps(result, indent=2))
print(json.dumps(result, indent=2))
assert status.get("status") in {"Done", "Completed"}
assert details.get("detailed_status", {}).get("status") == "SUCCESS"
assert metrics is not None

{
  "experiment": "004-training-real",
  "status": "Done",
  "job_id": "fb3606d6-93bc-4cbc-a060-db01dc537f24",
  "metrics": [
    {
      "metric": "val_loss",
      "values": {
        "0": 0.5858913064002991
      }
    },
    {
      "metric": "val_acc_1",
      "values": {
        "0": 0.875
      }
    },
    {
      "metric": "train_loss",
      "values": {
        "0": 0.7383504509925842
      }
    }
  ],
  "detailed_status": {
    "date": "10/19/2026",
    "time": "19:46:29",
    "message": "train action completed successfully for classification_pyt",
    "status": "SUCCESS"
  },
  "evidence_dir": "experiment-004-evidence"
}


## 7. Auditoria final e checklist de evidencias

Esta celula organiza os artefatos produzidos, cruza status/API/logs e gera um checklist final. Itens nao observados ficam como `NOT_CONFIRMED`; eles nao sao convertidos em sucesso por inferencia.

In [17]:
log_text = logs.text if hasattr(logs, "text") else ""
details = next(iter(status.get("job_details", {}).values()), {})
layout_tokens = {token: (token in log_text) for token in ["images_train", "images_test", "classes.txt"]}
checks = {
    "login_and_health": "readiness" in health,
    "workspace_validated": workspace.get("id") == WORKSPACE_ID,
    "dataset_registered": bool(matches),
    "schema_validated": schema.get("type") == "object",
    "base_experiment_validated": preflight.get("base_experiment_found", False),
    "one_gpu_configured": specs.get("train", {}).get("num_gpus") == 1,
    "one_epoch_configured": specs.get("train", {}).get("num_epochs") == 1,
    "dataset_root_configured": specs.get("dataset", {}).get("root_dir") == DATASET_URI,
    "job_terminal_success": status.get("status") in {"Done", "Completed"},
    "tao_success_message": details.get("detailed_status", {}).get("status") == "SUCCESS",
    "metrics_returned": bool(details.get("kpi")),
    "images_train_observed_in_logs": layout_tokens["images_train"],
    "classes_file_observed_in_logs": layout_tokens["classes.txt"],
}
report = {"experiment": "004-training-real", "job_id": JOB_ID, "status": "PASSED" if all(checks.values()) else "REVIEW_REQUIRED", "checks": checks, "layout_tokens": layout_tokens, "metrics": details.get("kpi", []), "evidence_files": sorted(path.name for path in EVIDENCE_DIR.iterdir())}
(EVIDENCE_DIR / "final-checklist.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
lines = ["| Check | Result |", "| --- | --- |"] + [f"| {name} | {'PASS' if value else 'NOT_CONFIRMED'} |" for name, value in checks.items()]
report_markdown = "# Experiment 004 - Checklist\n\n" + "\n".join(lines) + "\n\nStatus: **" + report["status"] + "**\n"
(EVIDENCE_DIR / "final-checklist.md").write_text(report_markdown, encoding="utf-8")
print("\n".join(lines))
print("\nFinal status:", report["status"])

| Check | Result |
| --- | --- |
| login_and_health | PASS |
| workspace_validated | PASS |
| dataset_registered | PASS |
| schema_validated | PASS |
| base_experiment_validated | PASS |
| one_gpu_configured | PASS |
| one_epoch_configured | PASS |
| dataset_root_configured | PASS |
| job_terminal_success | PASS |
| tao_success_message | PASS |
| metrics_returned | PASS |
| images_train_observed_in_logs | PASS |
| classes_file_observed_in_logs | PASS |

Final status: PASSED
